# Comparison Tables for Paper

This notebook generates all tables comparing:
1. **True system** — the 20-reaction CRN being recovered
2. **LASSO (sparse-learning-CRN)** — fixed λ across T values; T=200 lambda sweep
3. **Bayesian spike-and-slab** — rate recovery and model selection

Each table is displayed inline and saved as a `.tex` file in this folder.

**L2 error** is computed in the 15-dimensional polynomial coefficient space for both methods,
making them directly comparable.

## 0. Setup

In [1]:
import sys, os
import numpy as np
import pandas as pd

# Add repo root to path (notebook lives in writing/, repo root is ..)
REPO_ROOT   = os.path.abspath('..')
sys.path.insert(0, REPO_ROOT)          # gives access to CRN_Simulation and src/
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))  # direct import of parsing

SPARSE_RUNS  = os.path.join(REPO_ROOT, 'sparse_runs')
RESULTS_DIR  = os.path.join(REPO_ROOT, 'results')
DATA_DIR     = os.path.join(REPO_ROOT, 'data')
WRITING_DIR  = os.path.abspath('.')   # this notebook's folder = writing/

NETWORK_FILE  = os.path.join(DATA_DIR, 'example5_network.json')

# LASSO settings
FIXED_LAMBDA  = 0.01
SWEEP_LAMBDAS = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
T_VALUES      = [100, 200, 400]

def lam_str(lam): return str(lam).replace('.', 'p')
def sparse_run_dir(T, lam):
    return os.path.join(SPARSE_RUNS, f'example5_T{T}_lam{lam_str(lam)}')
def mcmc_result_dir(T):
    return os.path.join(RESULTS_DIR, f'example5_T{T}')

print('Repo root:', REPO_ROOT)


Repo root: /sessions/epic-nice-planck/mnt/BayesCRNInference


## 1. Load Network and Define Helpers

In [2]:
from src.parsing import load_reaction_network

result = load_reaction_network(NETWORK_FILE)
# Unpack return tuple
rxn_names_list   = result[2]   # list of 20 sampled reaction name strings
k_names_list     = result[3]   # list of 20 k-parameter names, e.g. 'k22'
compatible_rxns  = result[9]   # dict: tuple(stoich_change) -> [candidate reaction indices]
species_names    = result[10]  # ['A','B','X','Y']
reactant_mat     = result[7]   # (4, 210) reactant stoichiometry for all 210 reactions
param_vals       = result[5]   # {'k22': 1.046, ...} true rates for the 20 active reactions

sp   = species_names
n_sp = len(sp)

# True rate lookup: reaction_index -> rate
true_rate_by_idx = {int(k[1:]): v for k, v in param_vals.items()}

# Reaction name lookup: reaction_index -> human-readable string
name_by_idx = {int(k[1:]): rxn_names_list[i].rstrip(':') for i, k in enumerate(k_names_list)}

# Basis function labels (order matches sparse-learning output)
basis_labels = ['1'] + sp + [sp[i]+'*'+sp[j] for i in range(n_sp) for j in range(i, n_sp)]

# Display column order: 1 | A^2 A | B^2 B | X^2 X | Y^2 Y | AB AX AY BX BY XY
col_order = ['1','A*A','A','B*B','B','X*X','X','Y*Y','Y','A*B','A*X','A*Y','B*X','B*Y','X*Y']
col_idx   = [basis_labels.index(c) for c in col_order]

print('Species:', sp)
print('Basis labels:', basis_labels)
print(f'Active reactions ({len(true_rate_by_idx)}):', true_rate_by_idx)


Reaction network loaded from /sessions/epic-nice-planck/mnt/BayesCRNInference/data/example5_network.json
  Species:   ['A', 'B', 'X', 'Y']
  Reactions: 20 sampled  |  210 total in full CRN
  Unique stoichiometric changes: 130
Species: ['A', 'B', 'X', 'Y']
Basis labels: ['1', 'A', 'B', 'X', 'Y', 'A*A', 'A*B', 'A*X', 'A*Y', 'B*B', 'B*X', 'B*Y', 'X*X', 'X*Y', 'Y*Y']
Active reactions (20): {22: 1.046377380209165, 53: 0.7166996443671432, 155: 0.5897219661598223, 199: 0.7676002433735416, 63: 1.0673280680928032, 24: 0.2803159923659085, 208: 0.906127060897544, 163: 0.4221298458375011, 29: 2.203110446845896, 70: 0.452756813424444, 109: 1.2593443372879236, 87: 1.9789114814591777, 72: 1.5712998768102708, 204: 0.4886915332391283, 83: 1.0205988796694354, 5: 0.6781716941068556, 149: 0.4167214425597647, 164: 0.6456419355009646, 54: 1.2782894211721718, 18: 1.5168073835123705}


In [3]:
# ── Helper functions ──────────────────────────────────────────────────────────

def poly_vector_from_rates(change, rate_by_param_idx):
    """
    Build 15-dim polynomial coefficient vector for one stoichiometric channel.

    change             : list/tuple of 4 ints (stoichiometric change)
    rate_by_param_idx  : dict {param_index: rate} — estimated or true rates
                         Param_index i maps to compatible_rxns[tuple(change)][i]

    Uses the stochastic propensity expansion:
      unimolecular  A      ->  + rate * A
      bimolecular   A+B    ->  + rate * A*B
      bimolecular   2A     ->  + rate/2 * A^2  -  rate/2 * A
      zeroth-order  ∅      ->  + rate * 1
    """
    c    = np.zeros(len(basis_labels))
    key  = tuple(int(x) for x in change)
    if key not in compatible_rxns:
        return c
    candidates = compatible_rxns[key]
    for param_i, rate in rate_by_param_idx.items():
        if abs(rate) < 1e-12:
            continue
        rxn_idx = candidates[param_i]
        rv      = reactant_mat[:, rxn_idx]          # reactant stoich vector
        nz      = np.nonzero(rv)[0]
        if len(nz) == 0:
            c[basis_labels.index('1')] += rate
        elif len(nz) == 1:
            i, m = nz[0], int(rv[nz[0]])
            if m == 1:
                c[basis_labels.index(sp[i])] += rate
            elif m == 2:
                c[basis_labels.index(sp[i]+'*'+sp[i])] += rate / 2.0
                c[basis_labels.index(sp[i])]            -= rate / 2.0
        elif len(nz) == 2:
            c[basis_labels.index(sp[nz[0]]+'*'+sp[nz[1]])] += rate
    return c


def true_poly_vector(change):
    """True polynomial coefficient vector from known rates."""
    key = tuple(int(x) for x in change)
    if key not in compatible_rxns:
        return np.zeros(len(basis_labels))
    candidates = compatible_rxns[key]
    rate_by_param = {i: true_rate_by_idx.get(rxn, 0.0)
                     for i, rxn in enumerate(candidates)}
    return poly_vector_from_rates(change, rate_by_param)


def read_channel_info(run_dir):
    """Returns (changes, counts) from channel_info.txt."""
    path = os.path.join(run_dir, 'output', 'channel_info.txt')
    if not os.path.exists(path):
        return None, None
    with open(path) as f:
        lines = [l.strip() for l in f.read().strip().splitlines() if l.strip()]
    n_ch = int(lines[0].split()[0])
    changes = [list(map(int, lines[i+1].split())) for i in range(n_ch)]
    counts  = list(map(int, lines[n_ch+1].split()))
    return changes, counts


def load_omega(run_dir, ch_idx):
    """Load sparse-learning coefficient vector for channel ch_idx."""
    path = os.path.join(run_dir, 'output', f'omega_vec_for_channel_{ch_idx}.txt')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        parts = f.read().strip().split()
    n = int(parts[0])
    return np.array([float(x) for x in parts[1:n+1]])


def lasso_poly_l2(run_dir, changes, ref_changes=None):
    """
    Compute polynomial-space L2 error for each channel in run_dir.
    ref_changes : channel list from a reference run (e.g. T=100).
                  If provided, returns one L2 per ref_changes entry (NaN if missing).
    Returns dict: change_tuple -> L2
    """
    l2_by_change = {}
    for ch_idx, change in enumerate(changes):
        omega = load_omega(run_dir, ch_idx)
        if omega is None:
            continue
        # omega is in basis_labels order; reorder to col_order for consistency
        true_c = true_poly_vector(change)
        l2     = np.linalg.norm(omega - true_c)
        l2_by_change[tuple(change)] = l2
    return l2_by_change


def load_mcmc_summary(T):
    """Load mcmc_summary.xlsx for a given T. Returns DataFrame or None."""
    path = os.path.join(mcmc_result_dir(T), 'mcmc_summary.xlsx')
    if not os.path.exists(path):
        return None
    return pd.read_excel(path)


def mcmc_poly_l2(df_mcmc, changes):
    """
    Compute polynomial-space L2 error for each channel from MCMC summary.
    Returns dict: change_tuple -> L2
    """
    # Uses count_to_change_ref (built from T=100 channel info; channels are same across T)
    l2_by_change = {}
    for run_idx, grp in df_mcmc.groupby('Run_Index'):
        cnt    = grp['Count'].iloc[0]
        # find stoich change by matching count to the reference channel list
        change = count_to_change_ref.get(cnt)
        if change is None:
            continue
        key = tuple(change)
        # build rate_by_param_idx from posterior means
        rate_by_param = {int(row['Param_Index']): row['Mean']
                         for _, row in grp.iterrows()}
        est_c  = poly_vector_from_rates(change, rate_by_param)
        true_c = true_poly_vector(change)
        l2_by_change[key] = np.linalg.norm(est_c - true_c)
    return l2_by_change


print('Helpers defined.')

Helpers defined.


In [4]:
# Load reference channel info (T=100 — establishes channel order and obs counts)
ref_dir = sparse_run_dir(100, FIXED_LAMBDA)
ref_changes, ref_counts = read_channel_info(ref_dir)
assert ref_changes is not None, f'Cannot find channel_info in {ref_dir}'

count_to_change_ref = {cnt: ch for ch, cnt in zip(ref_changes, ref_counts)}

print(f'Reference channels ({len(ref_changes)}):')
for ch, cnt in zip(ref_changes, ref_counts):
    print(f'  {ch}  obs={cnt}')

Reference channels (15):
  [-2, 0, 0, 0]  obs=60
  [-2, 0, 1, 1]  obs=129
  [-2, 1, 0, 0]  obs=230
  [0, -2, 1, 0]  obs=615
  [0, 0, -1, 1]  obs=138
  [0, 0, 0, -1]  obs=310
  [0, 0, 0, 1]  obs=184
  [0, 0, 1, -1]  obs=136
  [0, 1, -1, 0]  obs=427
  [0, 1, -1, 1]  obs=317
  [0, 1, 0, -1]  obs=326
  [0, 1, 0, 0]  obs=378
  [0, 2, 0, 0]  obs=64
  [1, -1, 0, 0]  obs=580
  [1, 0, 0, 0]  obs=255


In [5]:
# Build run_to_change: stable Run_Index → stoich change (works across all T)
# count_to_change_ref only works for T=100; T=200/400 have larger event counts.
df100_ref = load_mcmc_summary(100)
run_to_change = {}
if df100_ref is not None:
    for run_idx, grp in df100_ref.groupby('Run_Index'):
        cnt    = grp['Count'].iloc[0]
        change = count_to_change_ref.get(cnt)
        if change is not None:
            run_to_change[int(run_idx)] = change
print(f'run_to_change: {len(run_to_change)} channels mapped via Run_Index')

# Re-define mcmc_poly_l2 to use run_idx (not count) for lookup
def mcmc_poly_l2(df_mcmc, changes):
    """Compute polynomial-space L2 error for each channel from MCMC summary."""
    l2_by_change = {}
    for run_idx, grp in df_mcmc.groupby('Run_Index'):
        change = run_to_change.get(int(run_idx))
        if change is None:
            continue
        key = tuple(change)
        rate_by_param = {int(row['Param_Index']): row['Mean']
                         for _, row in grp.iterrows()}
        est_c  = poly_vector_from_rates(change, rate_by_param)
        true_c = true_poly_vector(change)
        l2_by_change[key] = np.linalg.norm(est_c - true_c)
    return l2_by_change

print('mcmc_poly_l2 redefined (uses Run_Index mapping).')


run_to_change: 15 channels mapped via Run_Index
mcmc_poly_l2 redefined (uses Run_Index mapping).


## Table 1 — True System Description

The 20 active reactions that make up the true network, grouped by stoichiometric channel.
The full candidate CRN contains 210 possible reactions over 4 species with polynomial order 2.

In [6]:
def fmt_reaction(name_str):
    """Convert 'A_to_A+B' style string to 'A → A+B' style."""
    s = name_str.replace('Empty', '∅').replace('2A','2A').replace('_to_',' → ')
    return s

def fmt_change(change):
    parts = []
    for s, v in zip(sp, change):
        if v != 0:
            sgn = '+' if v > 0 else ''
            parts.append(f'{sgn}{v}{s}')
    return ', '.join(parts) if parts else '0'

# Build table rows
rows = []
for ch_idx, (change, obs) in enumerate(zip(ref_changes, ref_counts)):
    key = tuple(change)
    candidates = compatible_rxns.get(key, [])
    active = [(i, rxn_idx) for i, rxn_idx in enumerate(candidates)
              if true_rate_by_idx.get(rxn_idx, 0) > 0]
    for param_i, rxn_idx in active:
        rate = true_rate_by_idx[rxn_idx]
        name = fmt_reaction(name_by_idx.get(rxn_idx, f'rxn_{rxn_idx}'))
        rows.append({
            'Ch': ch_idx,
            'Change': fmt_change(change),
            'Reaction': name,
            'k (true)': f'{rate:.4f}',
            'Obs (T=100)': obs,
        })

df_true = pd.DataFrame(rows)
# Suppress repeated Ch/Change/Obs for multi-reaction channels
df_display = df_true.copy()
for col in ['Ch','Change','Obs (T=100)']:
    df_display[col] = df_display[col].where(~df_display.duplicated(subset=['Ch']), other='')

print(df_display.to_string(index=False))

Ch        Change  Reaction k (true) Obs (T=100)
 0           -2A    2A → ∅   0.4528          60
 1 -2A, +1X, +1Y  2A → X+Y   1.0206         129
 2      -2A, +1B    2A → B   1.5713         230
 3      -2B, +1X    2B → X   1.9789         615
 4      -1X, +1Y  X+Y → 2Y   0.4887         138
 5           -1Y   A+Y → A   0.5897         310
             -1Y   X+Y → X   0.7676         310
 6           +1Y   A → A+Y   0.2803         184
                    Y → 2Y   1.0673            
 7      +1X, -1Y A+Y → A+X   0.6456         136
 8      +1B, -1X  2X → B+X   1.2593         427
                 A+X → A+B   0.4167            
 9 +1B, -1X, +1Y   X → B+Y   1.2783         317
10      +1B, -1Y A+Y → A+B   0.4221         326
                 X+Y → B+X   0.9061            
11           +1B   A → A+B   1.0464         378
                   X → B+X   0.7167            
12           +2B    ∅ → 2B   0.6782          64
13      +1A, -1B     B → A   2.2031         580
14           +1A    A → 2A   1.5168     

In [7]:
# Save Table 1 as LaTeX — clean version
def fmt_rxn_latex(name_str):
    """Convert 'A_to_A+B' to LaTeX math string."""
    s = name_str.replace('Empty', r'\emptyset')
    left, right = s.split('_to_')
    return f'${left} \\to {right}$'

def fmt_change_latex(change):
    parts = []
    for s, v in zip(sp, change):
        if v != 0:
            sgn = '+' if v > 0 else ''
            parts.append(f'${sgn}{v}{s}$')
    return ', '.join(parts) if parts else '$0$'

lines = []
lines.append(r'% Table 1 — True CRN system description')
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\caption{%')
lines.append(r'  The true 20-reaction CRN over species $A,B,X,Y$.')
lines.append(r'  Channels group reactions sharing the same net stoichiometric change.')
lines.append(r'  Both the sparse-learning and Bayesian methods infer one independent')
lines.append(r'  propensity per channel; within each channel up to 5 bimolecular')
lines.append(r'  mass-action reactions are candidates.')
lines.append(r'  Obs\ $=$ observed firings in the $T{=}100$ trajectory set.')
lines.append(r'}')
lines.append(r'\label{tab:true-system}')
lines.append(r'\begin{tabular}{rlllr}')
lines.append(r'\toprule')
lines.append(r'\textbf{Ch} & \textbf{Change} & \textbf{Reaction} & \textbf{Rate $k$} & \textbf{Obs} \\')
lines.append(r'\midrule')

prev_ch = None
for _, row in df_true.iterrows():
    ch_i   = row['Ch']
    ch     = ref_changes[ch_i]
    k_val  = float(row['k (true)'])
    obs    = row['Obs (T=100)']
    # Find the reaction index that has this rate
    candidates = compatible_rxns[tuple(ch)]
    rxn_idx = next((r for r in candidates if abs(true_rate_by_idx.get(r, 0) - k_val) < 1e-4), None)
    rxn_lat = fmt_rxn_latex(name_by_idx[rxn_idx]) if rxn_idx and rxn_idx in name_by_idx else '---'
    ch_str  = str(ch_i) if ch_i != prev_ch else ''
    chg_str = fmt_change_latex(ch) if ch_i != prev_ch else ''
    obs_str = str(obs) if ch_i != prev_ch else ''
    lines.append(f'{ch_str} & {chg_str} & {rxn_lat} & {k_val:.4f} & {obs_str} \\\\')
    prev_ch = ch_i

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

tex1 = '\n'.join(lines)
out1 = os.path.join(WRITING_DIR, 'table1_true_system.tex')
with open(out1, 'w') as f:
    f.write(tex1)
print(f'Saved: {out1}')
print(tex1)


Saved: /sessions/epic-nice-planck/mnt/BayesCRNInference/writing/table1_true_system.tex
% Table 1 — True CRN system description
\begin{table}[htbp]
\centering
\caption{%
  The true 20-reaction CRN over species $A,B,X,Y$.
  Channels group reactions sharing the same net stoichiometric change.
  Both the sparse-learning and Bayesian methods infer one independent
  propensity per channel; within each channel up to 5 bimolecular
  mass-action reactions are candidates.
  Obs\ $=$ observed firings in the $T{=}100$ trajectory set.
}
\label{tab:true-system}
\begin{tabular}{rlllr}
\toprule
\textbf{Ch} & \textbf{Change} & \textbf{Reaction} & \textbf{Rate $k$} & \textbf{Obs} \\
\midrule
0 & $-2A$ & $2A \to \emptyset$ & 0.4528 & 60 \\
1 & $-2A$, $+1X$, $+1Y$ & $2A \to X+Y$ & 1.0206 & 129 \\
2 & $-2A$, $+1B$ & $2A \to B$ & 1.5713 & 230 \\
3 & $-2B$, $+1X$ & $2B \to X$ & 1.9789 & 615 \\
4 & $-1X$, $+1Y$ & $X+Y \to 2Y$ & 0.4887 & 138 \\
5 & $-1Y$ & $A+Y \to A$ & 0.5897 & 310 \\
 &  & $X+Y \to X$ & 0.76

## Table 2a — LASSO Fixed λ=0.01, All T Values

Polynomial-space L² error per channel for T=100, 200, 400.

In [8]:
# Compute L2 for each T
lasso_l2_fixed = {}   # T -> {change_tuple -> L2}
for T in T_VALUES:
    rdir = sparse_run_dir(T, FIXED_LAMBDA)
    ch, _ = read_channel_info(rdir)
    if ch is None:
        print(f'T={T}: results not found — skipping')
        lasso_l2_fixed[T] = {}
        continue
    lasso_l2_fixed[T] = lasso_poly_l2(rdir, ch)
    print(f'T={T}: {len(lasso_l2_fixed[T])} channels loaded')

# Build display DataFrame
rows2a = []
for ch, cnt in zip(ref_changes, ref_counts):
    key = tuple(ch)
    row = {'Ch': ref_changes.index(ch), 'Change': fmt_change(ch), 'Obs': cnt}
    for T in T_VALUES:
        l2 = lasso_l2_fixed[T].get(key, float('nan'))
        row[f'T={T}'] = f'{l2:.3f}' if not np.isnan(l2) else '—'
    rows2a.append(row)

df_2a = pd.DataFrame(rows2a)
df_2a.index = df_2a['Ch']
df_2a = df_2a.drop(columns='Ch')
print(df_2a.to_string())

T=100: 15 channels loaded
T=200: 15 channels loaded
T=400: 15 channels loaded
           Change  Obs  T=100  T=200  T=400
Ch                                         
0             -2A   60  0.487  0.374  0.317
1   -2A, +1X, +1Y  129  0.754  0.851  0.993
2        -2A, +1B  230  0.962  0.879  0.938
3        -2B, +1X  615  0.843  1.005  1.009
4        -1X, +1Y  138  0.226  0.292  0.320
5             -1Y  310  0.632  0.411  0.174
6             +1Y  184  0.484  0.451  0.178
7        +1X, -1Y  136  0.143  0.188  0.168
8        +1B, -1X  427  0.832  0.816  0.686
9   +1B, -1X, +1Y  317  0.360  0.308  0.182
10       +1B, -1Y  326  0.209  0.160  0.153
11            +1B  378  0.713  0.342  0.283
12            +2B   64  0.797  0.163  0.519
13       +1A, -1B  580  0.991  0.411  0.195
14            +1A  255  0.408  0.262  0.152


In [ ]:
# Table 2a (LASSO fixed lambda across T) removed — information is subsumed
# by Table 3 (Bayes vs LASSO L2 comparison). Cell kept as placeholder.
print('Table 2a suppressed.')


## Table 2b — LASSO T=200 Lambda Sweep

Polynomial L² error across λ values at fixed T=200.

In [10]:
# Compute L2 for each lambda at T=200
lasso_l2_sweep = {}  # lam -> {change_tuple -> L2}
for lam in SWEEP_LAMBDAS:
    rdir = sparse_run_dir(200, lam)
    ch, _ = read_channel_info(rdir)
    if ch is None:
        print(f'λ={lam}: results not found — skipping')
        lasso_l2_sweep[lam] = {}
        continue
    lasso_l2_sweep[lam] = lasso_poly_l2(rdir, ch)
    print(f'λ={lam}: {len(lasso_l2_sweep[lam])} channels loaded')

# Observation counts for T=200
ref_dir_200 = sparse_run_dir(200, FIXED_LAMBDA)
_, ref_counts_200 = read_channel_info(ref_dir_200)

rows2b = []
for ch, cnt100, cnt200 in zip(ref_changes, ref_counts,
                               ref_counts_200 if ref_counts_200 else ref_counts):
    key = tuple(ch)
    row = {'Ch': ref_changes.index(ch), 'Change': fmt_change(ch), 'Obs': cnt200}
    for lam in SWEEP_LAMBDAS:
        l2 = lasso_l2_sweep[lam].get(key, float('nan'))
        # Bold minimum
        row[f'λ={lam}'] = l2
    rows2b.append(row)

df_2b = pd.DataFrame(rows2b).set_index('Ch')

# Display with formatting
df_2b_display = df_2b.copy()
for col in [f'λ={lam}' for lam in SWEEP_LAMBDAS]:
    df_2b_display[col] = df_2b_display[col].apply(
        lambda v: f'{v:.3f}' if not np.isnan(v) else '—')
print(df_2b_display.to_string())

λ=0.001: 15 channels loaded
λ=0.005: 15 channels loaded
λ=0.01: 15 channels loaded
λ=0.05: 15 channels loaded
λ=0.1: 15 channels loaded
λ=0.5: 15 channels loaded
           Change   Obs λ=0.001 λ=0.005 λ=0.01 λ=0.05  λ=0.1  λ=0.5
Ch                                                                 
0             -2A   115   1.087   0.431  0.374  0.294  0.251  0.245
1   -2A, +1X, +1Y   263   1.477   0.919  0.851  0.632  0.570  0.556
2        -2A, +1B   452   0.919   0.881  0.879  0.866  0.832  0.833
3        -2B, +1X  1185   1.215   1.044  1.005  0.693  0.953  1.032
4        -1X, +1Y   252   1.867   0.763  0.292  0.083  0.029  0.073
5             -1Y   626   0.750   0.575  0.411  0.078  0.057  0.201
6             +1Y   398   0.540   0.378  0.451  0.481  0.648  1.124
7        +1X, -1Y   264   0.388   0.234  0.188  0.078  0.038  0.120
8        +1B, -1X   860   1.153   0.993  0.816  0.695  0.668  0.690
9   +1B, -1X, +1Y   603   0.594   0.465  0.308  0.155  0.322  1.315
10       +1B, -1Y   62

In [ ]:
# Save Table 2b as LaTeX — bold the minimum L2 per row, no Ch column
lines = []
lines.append(r'% Table 2b — LASSO lambda sweep at T=200')
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\small')
lines.append(r'\caption{%')
lines.append(r'  Polynomial-space $L^2$ error for sparse-learning-CRN at $T{=}200$')
lines.append(r'  across regularisation strengths $\lambda$.')
lines.append(r'  Obs\ $=$ observed event count in the $T{=}200$ trajectory.')
lines.append(r'  \textbf{Bold}: minimum $L^2$ per channel.')
lines.append(r'  No single $\lambda$ minimises error across all channels simultaneously,')
lines.append(r'  illustrating the fundamental limitation of global regularisation.')
lines.append(r'}')
lines.append(r'\label{tab:lasso-lambda-sweep}')
n_lam = len(SWEEP_LAMBDAS)
lines.append(r'\begin{tabular}{lr' + 'r'*n_lam + r'}')
lines.append(r'\toprule')
lam_headers = ' & '.join([f'$\\lambda={lam}$' for lam in SWEEP_LAMBDAS])
lines.append(rf'\textbf{{Change}} & \textbf{{Obs}} & {lam_headers} \\')
lines.append(r'\midrule')

for ch, cnt200 in zip(ref_changes,
                       ref_counts_200 if ref_counts_200 else ref_counts):
    key = tuple(ch)
    l2s = [lasso_l2_sweep[lam].get(key, float('nan')) for lam in SWEEP_LAMBDAS]
    min_l2 = min((v for v in l2s if not np.isnan(v)), default=float('nan'))
    cells = []
    for v in l2s:
        s = f'{v:.3f}' if not np.isnan(v) else '---'
        if not np.isnan(v) and abs(v - min_l2) < 1e-6:
            s = r'\textbf{' + s + r'}'
        cells.append(s)
    chg = fmt_change_latex(ch)
    lines.append(f'{chg} & {cnt200} & {" & ".join(cells)} \\\\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

tex2b = '\n'.join(lines)
out2b = os.path.join(WRITING_DIR, 'table2b_lasso_lambda_sweep.tex')
with open(out2b, 'w') as f:
    f.write(tex2b)
print(f'Saved: {out2b}')


## Table 3 — Bayesian Spike-and-Slab Results

For each channel and each T value (loaded if available): posterior mean rates, Prob(off),
95% credible intervals, and **polynomial-space L² error** (same metric as Tables 2a/2b).

In [12]:
# Load MCMC summaries for all available T values
mcmc_dfs = {}
for T in T_VALUES:
    df = load_mcmc_summary(T)
    if df is not None:
        mcmc_dfs[T] = df
        print(f'T={T}: loaded {len(df)} parameter rows')
    else:
        print(f'T={T}: mcmc_summary.xlsx not found — will be marked as pending')

# Compute polynomial L2 for each available T
mcmc_l2 = {}  # T -> {change_tuple -> L2}
for T, df in mcmc_dfs.items():
    mcmc_l2[T] = mcmc_poly_l2(df, ref_changes)
    print(f'T={T}: computed L2 for {len(mcmc_l2[T])} channels')

T=100: loaded 51 parameter rows


T=200: loaded 51 parameter rows
T=400: loaded 51 parameter rows
T=100: computed L2 for 15 channels
T=200: computed L2 for 15 channels
T=400: computed L2 for 15 channels


In [13]:
# ── Summary table: one row per channel, L2 columns for each T ──────────────

rows3 = []
for ch_i, (ch, cnt) in enumerate(zip(ref_changes, ref_counts)):
    key = tuple(ch)
    row = {'Ch': ch_i, 'Change': fmt_change(ch), 'Obs (T=100)': cnt}
    for T in T_VALUES:
        l2 = mcmc_l2.get(T, {}).get(key, float('nan'))
        row[f'Bayes L2 T={T}'] = f'{l2:.3f}' if not np.isnan(l2) else '(running)'
    # Also add LASSO L2 at fixed lambda for direct comparison
    for T in T_VALUES:
        l2 = lasso_l2_fixed.get(T, {}).get(key, float('nan'))
        row[f'LASSO L2 T={T}'] = f'{l2:.3f}' if not np.isnan(l2) else '—'
    rows3.append(row)

df_3 = pd.DataFrame(rows3).set_index('Ch')

# Interleave columns: for each T, show Bayes then LASSO
cols_ordered = ['Change', 'Obs (T=100)']
for T in T_VALUES:
    cols_ordered += [f'Bayes L2 T={T}', f'LASSO L2 T={T}']
print(df_3[cols_ordered].to_string())

           Change  Obs (T=100) Bayes L2 T=100 LASSO L2 T=100 Bayes L2 T=200 LASSO L2 T=200 Bayes L2 T=400 LASSO L2 T=400
Ch                                                                                                                      
0             -2A           60          0.008          0.487          0.021          0.374          0.008          0.317
1   -2A, +1X, +1Y          129          0.060          0.754          0.044          0.851          0.008          0.993
2        -2A, +1B          230          0.061          0.962          0.052          0.879          0.082          0.938
3        -2B, +1X          615          0.056          0.843          0.036          1.005          0.019          1.009
4        -1X, +1Y          138          0.045          0.226          0.012          0.292          0.006          0.320
5             -1Y          310          0.108          0.632          0.029          0.411          0.038          0.174
6             +1Y          184  

In [14]:
# Detailed rate table for T=100 (and other T if available)
def show_mcmc_detail(T):
    """Print per-channel, per-reaction detail for MCMC at a given T."""
    if T not in mcmc_dfs:
        print(f'T={T} not available yet.')
        return
    df = mcmc_dfs[T]

    rows = []
    for run_idx, grp in df.groupby('Run_Index'):
        change = run_to_change.get(int(run_idx))
        if change is None:
            continue
        key       = tuple(change)
        candidates = compatible_rxns.get(key, [])
        poly_l2   = mcmc_l2.get(T, {}).get(key, float('nan'))

        for _, r in grp.sort_values('Param_Index').iterrows():
            pi       = int(r['Param_Index'])
            rxn_idx  = candidates[pi] if pi < len(candidates) else None
            rxn_name = name_by_idx.get(rxn_idx, '—') if rxn_idx is not None else '—'
            rows.append({
                'Ch':       ref_changes.index(list(change)),
                'Change':   fmt_change(change),
                'Reaction': fmt_reaction(rxn_name),
                'True k':   f"{r['True_Theta']:.4f}",
                'Post mean': f"{r['Mean']:.4f}",
                'Post std':  f"{r['Std']:.4f}",
                '95% CI':   f"[{r['CI_lower']:.3f}, {r['CI_upper']:.3f}]",
                'Prob Off':  f"{r['Prob_Off']:.4f}",
                'Poly L2':  f'{poly_l2:.4f}' if not np.isnan(poly_l2) else '—',
            })

    df_detail = pd.DataFrame(rows)
    if df_detail.empty:
        print(f'T={T}: no matching channels found.')
        return df_detail
    for col in ['Ch','Change','Poly L2']:
        df_detail[col] = df_detail[col].where(~df_detail.duplicated(subset=['Ch']), other='')
    print(f'\n=== Bayesian Spike-and-Slab Detail — T={T} ===')
    print(df_detail.to_string(index=False))
    return df_detail

for T in T_VALUES:
    show_mcmc_detail(T)



=== Bayesian Spike-and-Slab Detail — T=100 ===
Ch        Change  Reaction True k Post mean Post std         95% CI Prob Off Poly L2
14           +1A         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000  0.0405
             +1A    A → 2A 1.5168    1.4763   0.0921 [1.302, 1.661]   0.0000  0.0405
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
11           +1B         — 0.0000    0.0003   0.0117 [0.000, 0.000]   0.9987  0.1685
                   A → A+B 1.0464    1.2137   0.1350 [0.948, 1.488]   0.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
                   X → B+X 0.7167    0.7369   0.0970 [0.553, 0.937]   0.0000        
                         — 0.0000    0.0000   0.0000 [0.000, 0.000]   1.0000        
 6           +1Y 

In [ ]:
# Save Table 3 (L2 comparison) as LaTeX — no Ch, no Obs columns
lines = []
lines.append(r'% Table 3 — Bayesian vs LASSO polynomial L2 comparison')
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\small')
lines.append(r'\caption{%')
lines.append(r'  Polynomial-space $L^2$ propensity error,')
lines.append(r'  $\|\hat{c} - c_{\mathrm{true}}\|_2$, for each stoichiometric channel')
lines.append(r'  across trajectory lengths $T$.')
lines.append(r'  \textbf{Bold}: lower error for each $(\text{channel},\,T)$ pair.')
lines.append(r'  LASSO uses fixed $\lambda = ' + str(FIXED_LAMBDA) + r'$.')
lines.append(r'}')
lines.append(r'\label{tab:comparison-l2}')
lines.append(r'\begin{tabular}{l|rr|rr|rr}')
lines.append(r'\toprule')
t_groups = ' & '.join([rf'\multicolumn{{2}}{{c}}{{$T={T}$}}' for T in T_VALUES])
lines.append(rf'\multicolumn{{1}}{{c}}{{}} & {t_groups} \\')
cmidrules = ''.join([rf'\cmidrule(lr){{{2+2*i}-{3+2*i}}}' for i in range(len(T_VALUES))])
lines.append(cmidrules)
sub_headers = ' & '.join(['Bayes & LASSO'] * len(T_VALUES))
lines.append(rf'\textbf{{Change}} & {sub_headers} \\')
lines.append(r'\midrule')

for ch in ref_changes:
    key = tuple(ch)
    chg = fmt_change_latex(ch)
    cells = []
    for T in T_VALUES:
        bl2 = mcmc_l2.get(T, {}).get(key, float('nan'))
        ll2 = lasso_l2_fixed.get(T, {}).get(key, float('nan'))
        bs = f'{bl2:.3f}' if not np.isnan(bl2) else r'\textit{(r)}'
        ls = f'{ll2:.3f}' if not np.isnan(ll2) else '---'
        if not np.isnan(bl2) and not np.isnan(ll2):
            if bl2 <= ll2:
                bs = r'\textbf{' + bs + r'}'
            else:
                ls = r'\textbf{' + ls + r'}'
        cells.append(f'{bs} & {ls}')
    lines.append(f'{chg} & {" & ".join(cells)} \\\\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

tex3 = '\n'.join(lines)
out3 = os.path.join(WRITING_DIR, 'table3_comparison_l2.tex')
with open(out3, 'w') as f:
    f.write(tex3)
print(f'Saved: {out3}')


In [16]:
# ── Table 4: Predator-Prey Bayesian Detail ────────────────────────────────────
# Requires: results/example4_T40/mcmc_summary.xlsx and data/example4_T40_trajectory.json

from src.parsing import load_trajectory, parse_trajectory

result4    = load_reaction_network(os.path.join(DATA_DIR, 'example4_network.json'))
rmat4      = result4[7]
uc4        = result4[8]
cr4        = result4[9]
sp4        = result4[10]
rnames4    = result4[2]
ridx4      = result4[6]
rvals4     = result4[5]
rnames_k4  = result4[3]
name4      = {int(gi): rvals4.get(rn, 0.0) for gi, rn in zip(ridx4, rnames_k4)}
rxn_net4   = {gi: ch for ch, rxns in cr4.items() for gi in rxns}

def side4(v, sp):
    parts = [s if int(vi)==1 else f'{int(vi)}{s}' for vi,s in zip(v,sp) if int(vi)!=0]
    return '+'.join(parts) if parts else r'\emptyset'

def rxn4_latex(gi):
    reac = rmat4[:, gi]
    prod = reac + np.array(rxn_net4.get(gi, [0]*4))
    return f'${side4(reac,sp4)}\\to {side4(prod,sp4)}$'

time4, state4 = load_trajectory(os.path.join(DATA_DIR, 'example4_T40_trajectory.json'))
_, jc4, _, _  = parse_trajectory(state4, time4, rmat4, uc4, cr4, verbose=False)
ci4 = {ch:i for i,ch in enumerate(uc4)}
def nev4(ch):
    ci = ci4.get(ch)
    return int(sum(cnts[ci] for cnts in jc4.values())) if ci is not None else 0

df4 = pd.read_excel(os.path.join(RESULTS_DIR, 'example4_T40', 'mcmc_summary.xlsx'))
cnt4_to_ch = {1933:(1,0), 190:(-1,0), 1843:(-1,1), 1837:(0,-1)}
r2ch4 = {int(ri): cnt4_to_ch[int(grp['Count'].iloc[0])]
         for ri,grp in df4.groupby('Run_Index') if int(grp['Count'].iloc[0]) in cnt4_to_ch}
mcmc4 = {(r2ch4[int(r['Run_Index'])], int(r['Param_Index'])): r
         for _,r in df4.iterrows() if int(r['Run_Index']) in r2ch4}

lasso4 = {
    (1,0):  r'$0.969\,A$',
    (-1,0): r'$0.018\,A + 0.036\,B^{\,\dagger}$',
    (-1,1): r'$0.010\,AB$',
    (0,-1): r'$0.497\,B$',
}
chlabel4 = {(1,0):r'$(+1,\;0)$', (-1,0):r'$(-1,\;0)$',
             (-1,1):r'$(-1,+1)$', (0,-1):r'$(0,\;-1)$'}

lines = []
lines.append(r'% Table 4 — Predator-prey Bayesian detail')
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\small')
lines.append(r'\caption{Predator-prey inference at $T=40$ (5{,}805 events total). '
             r'All candidate mass-action reactions for each stoichiometric channel $\xi_\ell$ are shown. '
             r'$k^*$: true rate (bold if active). '
             r'$\hat{k}$: posterior mean (95\% credible interval). '
             r'$\Pr(\text{off})$: spike-and-slab exclusion probability. '
             r'LASSO propensity at $\lambda=0.01$. '
             r'$^\dagger$~Mechanistically impossible under mass-action kinetics.}')
lines.append(r'\label{tab:predprey_detail}')
lines.append(r'\begin{tabular}{l l r r l r l}')
lines.append(r'\toprule')
lines.append(r'$\xi_\ell$ & Candidate reaction & $k^*$ & Events & '
             r'$\hat{k}$\;(95\% CI) & $\Pr(\text{off})$ & LASSO \\')
lines.append(r'\midrule')

for ch in [(1,0),(-1,0),(-1,1),(0,-1)]:
    cands = cr4.get(ch, [])
    nev   = nev4(ch)
    n_c   = len(cands)
    for pi, gi in enumerate(cands):
        true_k = name4.get(gi, 0.0)
        rxn    = rxn4_latex(gi)
        row    = mcmc4.get((ch, pi))
        ch_col = rf'\multirow{{{n_c}}}{{*}}{{{chlabel4[ch]}}}' if pi==0 else ''
        ev_col = str(nev) if pi==0 else ''
        k_str  = rf'\textbf{{{true_k:.4f}}}' if true_k>1e-9 else '$0$'
        if row is not None:
            poff = row['Prob_Off']
            if poff >= 0.99:
                post_str = '---'; poff_str = rf'${poff:.3f}$'
            else:
                lo,hi = row['CI_lower'],row['CI_upper']
                post_str = rf'${row["Mean"]:.4f}\;({lo:.4f},\,{hi:.4f})$'
                poff_str = rf'${poff:.3f}$'
        else:
            post_str = '---'; poff_str = '---'
        lasso_str = lasso4[ch] if true_k>1e-9 else ''
        lines.append(rf'{ch_col} & {rxn} & {k_str} & {ev_col} & {post_str} & {poff_str} & {lasso_str} \\')
    lines.append(r'\midrule')

lines[-1] = r'\bottomrule'
lines += [r'\end{tabular}', r'\end{table}']
tex4 = '\n'.join(lines)
out4 = os.path.join(WRITING_DIR, 'table_predprey_detail.tex')
with open(out4, 'w') as f:
    f.write(tex4)
print(f'Saved: {out4}')


Reaction network loaded from /sessions/epic-nice-planck/mnt/BayesCRNInference/data/example4_network.json
  Species:   ['A', 'B']
  Reactions: 4 sampled  |  30 total in full CRN
  Unique stoichiometric changes: 18


Saved: /sessions/epic-nice-planck/mnt/BayesCRNInference/writing/table_predprey_detail.tex


In [ ]:
# ── Figure 1: Predator-Prey Stochastic Trajectory ──────────────────────────
import json as _json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

with open(os.path.join(DATA_DIR, 'example4_T40_trajectory.json')) as _f:
    _traj = _json.load(_f)

_times  = _traj['time']
_states = _traj['states']
_A = [s[0] for s in _states]
_B = [s[1] for s in _states]

# Extend arrays so step plot reaches T=40
_te = _times + [40.0]
_Ae = _A + [_A[-1]]
_Be = _B + [_B[-1]]

_FS, _TS, _LW = 16, 14, 1.1   # axis label, tick, line width

_fig, _ax = plt.subplots(figsize=(8, 4))
_ax.step(_te, _Ae, where='post', color='#1f77b4', linewidth=_LW, label='Prey ($A$)')
_ax.step(_te, _Be, where='post', color='#d62728', linewidth=_LW, label='Predator ($B$)')

# Mark extinction
_ext = 39.758
_ax.axvline(x=_ext, color='#555555', linestyle=':', linewidth=1.2)
_ax.annotate('Extinction\n$t\\approx 39.8$',
             xy=(_ext, 40), xytext=(36.5, 140),
             fontsize=11, color='#555555', ha='center',
             arrowprops=dict(arrowstyle='->', color='#555555', lw=0.9))

_ax.set_xlabel('Time', fontsize=_FS, labelpad=6)
_ax.set_ylabel('Population count', fontsize=_FS, labelpad=6)
_ax.tick_params(axis='both', labelsize=_TS)
_ax.legend(fontsize=_TS, framealpha=0.9, loc='upper left')
_ax.set_xlim(0, 40)
_ax.set_ylim(bottom=0)
_ax.spines['top'].set_visible(False)
_ax.spines['right'].set_visible(False)

plt.tight_layout()
_out1 = os.path.join(WRITING_DIR, 'fig_predprey_traj.pdf')
_fig.savefig(_out1, bbox_inches='tight')
plt.close(_fig)
print('Saved:', _out1)


In [ ]:
# Save Tables 4/5/6 — Bayesian detail, predator-prey style
# Columns: Change | Obs | Reaction | True k* | k-hat (95% CI) | Pr(off)

def save_mcmc_detail_latex(T):
    if T not in mcmc_dfs:
        print('T=' + str(T) + ' not available.')
        return
    df_m = mcmc_dfs[T]

    # Obs counts for this T
    _, counts_T = read_channel_info(sparse_run_dir(T, FIXED_LAMBDA))
    obs_for_T = {tuple(ch): cnt for ch, cnt in zip(ref_changes, counts_T)} if counts_T else {}

    # Build change -> list of visible rows (in ref_changes order)
    change_to_run = {tuple(ch): ri for ri, ch in run_to_change.items()}
    channel_data = []   # list of (change, obs, [row, ...])
    for ch in ref_changes:
        key = tuple(ch)
        run_idx = change_to_run.get(key)
        if run_idx is None:
            continue
        grp = df_m[df_m['Run_Index'] == run_idx]
        if grp.empty:
            continue
        visible = [r for _, r in grp.sort_values('Param_Index').iterrows()
                   if not (r['True_Theta'] < 1e-6 and r['Prob_Off'] > 0.99)]
        if visible:
            channel_data.append((ch, obs_for_T.get(key, '?'), visible))

    lines = []
    lines.append('% Bayesian spike-and-slab detail T=' + str(T))
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    lines.append(r'\small')
    lines.append(
        r'\caption{Bayesian spike-and-slab posterior estimates, $T=' + str(T) + r'$.'
        r' Stoichiometric change $\xi_\ell$ and observed event count (Obs) are shown'
        r' once per channel. True rate $k^*$ is bold for active reactions.'
        r' $\hat{k}$ and 95\% credible interval are combined.'
        r' Candidates with $\Pr(\text{off})>0.99$ are omitted.}'
    )
    lines.append(r'\label{tab:bayes-detail-T' + str(T) + '}')
    lines.append(r'\begin{tabular}{lrlrll}')
    lines.append(r'\toprule')
    lines.append(
        r'\textbf{Change} & \textbf{Obs} & \textbf{Reaction}'
        r' & \textbf{True $k^*$} & $\hat{k}$\;(95\% CI) & $\Pr(\text{off})$ \\'
    )
    lines.append(r'\midrule')

    for ch_i, (ch, obs, rows) in enumerate(channel_data):
        if ch_i > 0:
            lines.append(r'\midrule')
        key   = tuple(ch)
        cands = compatible_rxns.get(key, [])
        chg   = fmt_change_latex(ch)
        n     = len(rows)

        for row_i, r in enumerate(rows):
            pi       = int(r['Param_Index'])
            true_k   = r['True_Theta']
            prob_off = r['Prob_Off']
            rxn_idx  = cands[pi] if pi < len(cands) else None
            rxn_name = name_by_idx.get(rxn_idx, '') if rxn_idx is not None else ''
            rxn_lat  = fmt_rxn_latex(rxn_name) if rxn_name else '---'
            ci_str   = ('[' + '{:.3f}'.format(r['CI_lower'])
                        + r',\,' + '{:.3f}'.format(r['CI_upper']) + ']')
            k_ci     = '{:.4f}'.format(r['Mean']) + r'\;' + ci_str
            tk_str   = (r'\textbf{' + '{:.4f}'.format(true_k) + '}'
                        if true_k > 1e-6 else '{:.4f}'.format(true_k))

            if row_i == 0:
                chg_cell = (r'\multirow{' + str(n) + r'}{*}{' + chg + '}'
                            if n > 1 else chg)
                obs_cell = (r'\multirow{' + str(n) + r'}{*}{' + str(obs) + '}'
                            if n > 1 else str(obs))
            else:
                chg_cell = ''
                obs_cell = ''

            lines.append(
                chg_cell + ' & ' + obs_cell + ' & ' + rxn_lat
                + ' & ' + tk_str
                + ' & ' + k_ci
                + ' & ' + '{:.4f}'.format(prob_off) + r' \\'
            )

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    tex = '\n'.join(lines)
    out = os.path.join(WRITING_DIR, 'table3b_bayes_detail_T' + str(T) + '.tex')
    with open(out, 'w') as fh:
        fh.write(tex)
    print('Saved: ' + out)

for T in T_VALUES:
    save_mcmc_detail_latex(T)


## Table S1 — Full Polynomial Coefficient Table (Supplement)

For each stoichiometric channel: True propensity polynomial and recovered coefficients
at T=100, 200, 400 (λ=0.01). All 15 basis function columns shown.

Saved as `table_supp_poly_coeffs.tex`. Requires `\\usepackage{rotating}` in preamble.
Use `\\begin{sidewaystable}` — do **not** use `pdflscape` + `table[p]`, which causes
the table to disappear when used as a float inside a landscape block.

In [18]:
# ── Supplement: full polynomial coefficient table ─────────────────────────
# Uses sidewaystable (rotating package) + resizebox(\textheight) for correct
# landscape rendering. Do NOT use pdflscape + table[p] — floats escape the
# landscape block and become invisible.

import os
import numpy as np

def build_poly_coeff_table(FIXED_LAMBDA=0.01):
    lam = FIXED_LAMBDA
    T_runs = [
        ('T=100', sparse_run_dir(100, lam)),
        ('T=200', sparse_run_dir(200, lam)),
        ('T=400', sparse_run_dir(400, lam)),
    ]

    # Column order: 1 | A² A | B² B | X² X | Y² Y | AB AX AY BX BY XY
    col_order = ['1','A*A','A','B*B','B','X*X','X','Y*Y','Y',
                 'A*B','A*X','A*Y','B*X','B*Y','X*Y']
    col_latex  = ['$1$','$A^2$','$A$','$B^2$','$B$','$X^2$','$X$','$Y^2$','$Y$',
                  '$AB$','$AX$','$AY$','$BX$','$BY$','$XY$']
    cidx = [basis_labels.index(c) for c in col_order]

    def get_row(c_vec):
        return [c_vec[i] for i in cidx]

    def fmt(v):
        return '0' if abs(v) < 5e-4 else f'{v:+.3f}'

    def chg_latex(change):
        parts = []
        for s, v in zip(sp, change):
            if v != 0:
                sgn = '+' if v > 0 else ''
                parts.append(f'${sgn}{v}{s}$')
        return ', '.join(parts) if parts else '$0$'

    # Load omega vectors for each T
    t_omegas = {}
    for tlabel, rdir in T_runs:
        ch_list, _ = read_channel_info(rdir)
        if ch_list is None:
            t_omegas[tlabel] = None
            continue
        omegas = {}
        for i, ch in enumerate(ch_list):
            omega = load_omega(rdir, i)
            if omega is not None:
                omegas[tuple(ch)] = omega
        t_omegas[tlabel] = omegas

    lines = []
    lines.append(r'% Supplement: polynomial coefficient table')
    lines.append(r'% Requires: \usepackage{rotating}, \usepackage{booktabs}')
    lines.append(r'% Use \resizebox{\textheight}{!} because the table is rotated')
    lines.append(r'')
    lines.append(r'\begin{sidewaystable}[p]')
    lines.append(r'\centering')
    lines.append(r'\caption{%')
    lines.append(rf'  Polynomial coefficient recovery by sparse-learning-CRN ($\lambda = {lam}$).')
    lines.append(r'  Each channel is identified by its stoichiometric change vector.')
    lines.append(r'  \emph{True}: exact stochastic propensity polynomial')
    lines.append(r'  (bimolecular same-species reactions expanded as')
    lines.append(r'  $A(A{-}1)/2 = \tfrac{1}{2}A^2 - \tfrac{1}{2}A$, bold).')
    lines.append(r'  Rows $T{=}100,200,400$: recovered coefficients.')
    lines.append(r'  Obs: firing events in $T{=}100$ trajectories.')
    lines.append(r'  Columns: constant $|$ quadratic/linear pairs per species $|$ cross terms.')
    lines.append(r'}')
    lines.append(r'\label{tab:poly-coeffs}')
    lines.append(r'\resizebox{\textheight}{!}{%')
    lines.append(r'\begin{tabular}{rrlr|r|rr|rr|rr|rr|rrrrrr}')
    lines.append(r'\toprule')
    # Header row 1
    lines.append(
        r'\multicolumn{4}{c}{} & & '
        r'\multicolumn{2}{c}{$A$} & '
        r'\multicolumn{2}{c}{$B$} & '
        r'\multicolumn{2}{c}{$X$} & '
        r'\multicolumn{2}{c}{$Y$} & '
        r'\multicolumn{6}{c}{cross terms} \\'
    )
    lines.append(r'\cmidrule(lr){7-8}\cmidrule(lr){9-10}\cmidrule(lr){11-12}\cmidrule(lr){13-14}\cmidrule(lr){15-20}')
    # Header row 2
    header = (
        r'\textbf{Ch} & \textbf{Obs} & \textbf{Change} & \textbf{Row} & '
        + ' & '.join(col_latex) + r' \\'
    )
    lines.append(header)
    lines.append(r'\midrule')

    for ch_i, (change, obs) in enumerate(zip(ref_changes, ref_counts)):
        key     = tuple(change)
        true_c  = true_poly_vector(change)
        true_row = get_row(true_c)

        # True row — bold nonzero entries
        true_cells = [str(ch_i), str(obs), chg_latex(change), r'\textit{True}']
        for v in true_row:
            if abs(v) > 5e-4:
                true_cells.append(r'\textbf{' + fmt(v) + r'}')
            else:
                true_cells.append('0')
        lines.append(' & '.join(true_cells) + r' \\')

        # T rows
        for tlabel, _ in T_runs:
            omegas = t_omegas.get(tlabel)
            if omegas is None or key not in omegas:
                omega = np.zeros(len(basis_labels))
                row_label = f'{tlabel} (n/a)'
            else:
                omega = omegas[key]
                row_label = tlabel
            cells = ['', '', '', row_label] + [fmt(v) for v in get_row(omega)]
            lines.append(' & '.join(cells) + r' \\')

        if ch_i < len(ref_changes) - 1:
            lines.append(r'\midrule')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'}')  # close resizebox
    lines.append(r'\end{sidewaystable}')

    return '\n'.join(lines)


tex_supp = build_poly_coeff_table(FIXED_LAMBDA)
out_supp = os.path.join(WRITING_DIR, 'table_supp_poly_coeffs.tex')
with open(out_supp, 'w') as f:
    f.write(tex_supp)
print(f'Saved: {out_supp}')
print(f'Lines: {tex_supp.count(chr(10))}')
# Show first few data lines as sanity check
for line in tex_supp.split('\n')[20:26]:
    print(line)


Saved: /sessions/epic-nice-planck/mnt/BayesCRNInference/writing/table_supp_poly_coeffs.tex
Lines: 101
\multicolumn{4}{c}{} & & \multicolumn{2}{c}{$A$} & \multicolumn{2}{c}{$B$} & \multicolumn{2}{c}{$X$} & \multicolumn{2}{c}{$Y$} & \multicolumn{6}{c}{cross terms} \\
\cmidrule(lr){7-8}\cmidrule(lr){9-10}\cmidrule(lr){11-12}\cmidrule(lr){13-14}\cmidrule(lr){15-20}
\textbf{Ch} & \textbf{Obs} & \textbf{Change} & \textbf{Row} & $1$ & $A^2$ & $A$ & $B^2$ & $B$ & $X^2$ & $X$ & $Y^2$ & $Y$ & $AB$ & $AX$ & $AY$ & $BX$ & $BY$ & $XY$ \\
\midrule
0 & 60 & $-2A$ & \textit{True} & 0 & \textbf{+0.226} & \textbf{-0.226} & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 &  &  & T=100 & -0.389 & +0.141 & -0.259 & -0.029 & +0.126 & 0 & -0.094 & -0.008 & 0 & +0.109 & +0.162 & -0.046 & -0.092 & +0.001 & +0.062 \\


In [ ]:
# ── Figure 2: Four-Species Stochastic Trajectory ────────────────────────────
with open(os.path.join(DATA_DIR, 'example5_T400_trajectory.json')) as _f:
    _traj5 = _json.load(_f)

_times5  = _traj5['time']
_states5 = _traj5['states']
_sp_vals = [[s[j] for s in _states5] for j in range(4)]

_sp_names  = ['$A$', '$B$', '$X$', '$Y$']
_sp_colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']

_te5  = _times5 + [400.0]
_sp_e = [v + [v[-1]] for v in _sp_vals]

_FS2, _TS2 = 15, 13

_fig2, _axes = plt.subplots(2, 2, figsize=(10, 5.5), sharex=True)
_axes = _axes.flatten()

for _j, (_ax2, _name, _color) in enumerate(zip(_axes, _sp_names, _sp_colors)):
    _ax2.step(_te5, _sp_e[_j], where='post', color=_color, linewidth=0.75, alpha=0.85)
    _ax2.set_ylabel(f'Count of {_name}', fontsize=_FS2)
    _ax2.tick_params(axis='both', labelsize=_TS2)
    _ax2.set_xlim(0, 400)
    _ax2.set_ylim(bottom=0)
    _ax2.spines['top'].set_visible(False)
    _ax2.spines['right'].set_visible(False)
    for _Tc, _ls, _al in [(100, '--', 0.65), (200, ':', 0.65)]:
        _ax2.axvline(x=_Tc, color='#666666', linestyle=_ls, linewidth=0.9, alpha=_al)

_axes[2].set_xlabel('Time', fontsize=_FS2, labelpad=5)
_axes[3].set_xlabel('Time', fontsize=_FS2, labelpad=5)

_leg_handles = [
    Line2D([0],[0], color='#666666', linestyle='--', linewidth=0.9, label='$T=100$'),
    Line2D([0],[0], color='#666666', linestyle=':',  linewidth=0.9, label='$T=200$'),
]
_axes[0].legend(handles=_leg_handles, fontsize=_TS2-1, framealpha=0.9, loc='upper right')

plt.tight_layout(h_pad=1.0, w_pad=1.5)
_out2 = os.path.join(WRITING_DIR, 'fig_example5_traj.pdf')
_fig2.savefig(_out2, bbox_inches='tight')
plt.close(_fig2)
print('Saved:', _out2)


---
## Summary of saved LaTeX files

| File | Description |
|------|-------------|
| `table1_true_system.tex` | True 20-reaction CRN description |
| `table2a_lasso_fixed_lambda.tex` | LASSO L² vs T at fixed λ |
| `table2b_lasso_lambda_sweep.tex` | LASSO L² vs λ at T=200 (bold = per-channel min) |
| `table3_comparison_l2.tex` | Bayes vs LASSO polynomial L² (all T) |
| `table3b_bayes_detail_T{N}.tex` | Detailed rate estimates + Prob(off) per T |
| `table_supp_poly_coeffs.tex` | **Supplement**: full 15-column polynomial coefficients |

**Preamble requirements:** `booktabs`, `rotating` (for `sidewaystable`).

To include in Overleaf: `\\input{table1_true_system}` etc.
For the supplement table: `\\input{table_supp_poly_coeffs}` inside `\\begin{appendices}`.


In [ ]:
# Sync all generated tables and figures to the paper directory
import shutil

PAPER_DIR = os.path.join(REPO_ROOT, 'Chemical_Reaction_Network_Inference')

tables = [
    'table1_true_system.tex',
    'table2b_lasso_lambda_sweep.tex',
    'table3_comparison_l2.tex',
    'table3b_bayes_detail_T100.tex',
    'table3b_bayes_detail_T200.tex',
    'table3b_bayes_detail_T400.tex',
    'table_predprey_detail.tex',
]
figures = [
    'fig_predprey_traj.pdf',
    'fig_example5_traj.pdf',
]

for fname in tables + figures:
    src = os.path.join(WRITING_DIR, fname)
    dst = os.path.join(PAPER_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print('  Copied', fname)
    else:
        print('  MISSING:', fname)
print('Sync complete.')
